In [20]:
import pandas as pd
import numpy as np
import os

In [21]:
df_train = pd.read_csv("../data/original/customer_churn_dataset-training-master.csv")

### 1. Fix Leakage: Calculate bins on entire dataset to avoid relative bins per split
By calling `qcut` on the entire training set, we define what "High Spend" means globally, so we can apply the exact same thresholds sequentially over every small batch without distribution shift.

In [22]:
# Compute the quantiles for "Total Spend" across the FULL training set
_, spend_bins = pd.qcut(df_train["Total Spend"], q=4, retbins=True)
spend_bins[0] = -np.inf
spend_bins[-1] = np.inf
spend_labels = ["Low", "Medium", "High", "Very High"]

tenure_bins = [0, 12, 24, 36, 100]
tenure_labels = ["<1yr", "1-2yr", "2-3yr", "3+yr"]

### 2. Feature Engineering & Preprocessing Function
A reusable piece of logic that sets the `CustomerID` index, casts columns, extracts features, and creates the dummy variables AT THE VERY END.

In [23]:
def preprocess_data(df, spend_bins, spend_labels, tenure_bins, tenure_labels):
    # Explicitly move CustomerID to index so it isn't treated as a training feature
    df = df.set_index("CustomerID")

    df = df.dropna()
    
    # Type conversions
    int_columns = ["Age", "Tenure", "Support Calls", "Last Interaction"]
    for col in int_columns:
        df[col] = df[col].astype("int64")
        
    category_features = ["Gender", "Subscription Type", "Contract Length"]
    for feature in category_features:
        df[feature] = df[feature].astype("category")
        
    # Feature Engineering
    df['Tenure_Age_Ratio'] = df['Tenure'] / (df['Age'] + 1)
    df["Spend_per_Usage"] = df["Total Spend"] / (df["Usage Frequency"] + 1)
    df["Support_Calls_per_Tenure"] = df["Support Calls"] / (df["Tenure"] + 1)
    
    # Create segments 
    df["Spending_Group"] = pd.cut(df["Total Spend"], bins=spend_bins, labels=spend_labels)
    df["Tenure_Group"] = pd.cut(df["Tenure"], bins=tenure_bins, labels=tenure_labels)
    
    # Expand category_features array to include the new segments
    category_features.extend(["Spending_Group", "Tenure_Group"])
    
    # Get Dummies
    df_processed = pd.get_dummies(df, columns=category_features, drop_first=True)
    
    return df_processed

### 3. Iteratively Preprocess All 10 Splits
Pass all newly defined bins into your `preprocess_data` function for every split and save.

In [24]:
out_dir = "../data/processed"
os.makedirs(out_dir, exist_ok=True)

for i in range(1, 11):
    df_raw = pd.read_csv(f"../data/raw/train_period_{i}.csv")
    
    df_processed = preprocess_data(df_raw, spend_bins, spend_labels, tenure_bins, tenure_labels)
    
    out_path = os.path.join(out_dir, f"df_processed_period_{i}.csv")
    df_processed.to_csv(out_path, index=True) # Index has CustomerID now
    print(f"Exported processed data to {out_path}")

Exported processed data to ../data/processed/df_processed_period_1.csv
Exported processed data to ../data/processed/df_processed_period_2.csv
Exported processed data to ../data/processed/df_processed_period_3.csv
Exported processed data to ../data/processed/df_processed_period_4.csv
Exported processed data to ../data/processed/df_processed_period_5.csv
Exported processed data to ../data/processed/df_processed_period_6.csv
Exported processed data to ../data/processed/df_processed_period_7.csv
Exported processed data to ../data/processed/df_processed_period_8.csv
Exported processed data to ../data/processed/df_processed_period_9.csv
Exported processed data to ../data/processed/df_processed_period_10.csv
